# Trabajo Práctico 2 - Grupo 02

### Modelo Random Forest

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen

El objetivo es entrenar un modelo de Random Forest sobre dos representaciones de texto (TF-IDF y BoW) con búsqueda de hiperparámetros, comparando su desempeño contra el baseline de Naive Bayes (~0.66 F1-macro en Kaggle).

Random Forest es un ensemble de árboles de decisión entrenados sobre subconjuntos aleatorios del data y de las features. Al promediar muchos árboles distintos reduce la varianza y generaliza mejor que un árbol solo.

## Importación e instalación de dependencias


In [4]:
!pip install -q spacy
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 79.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV

In [6]:
from common.preprocessing import clean_classical
from common.data_utils import get_split, make_tfidf, make_bow, SEED
from common.evaluation import evaluate
from common.io_utils import save_model, load_model, make_submission

np.random.seed(SEED)

## Carga de datos y preprocesamiento

In [8]:
train = pd.read_csv("/content/train.csv")
test  = pd.read_csv("/content/test.csv")
X_train_raw, X_val_raw, y_train, y_val = get_split(train)

In [9]:
print("Preprocesando")
X_train = np.array([clean_classical(t) for t in X_train_raw])
X_val   = np.array([clean_classical(t) for t in X_val_raw])
X_test  = np.array([clean_classical(t) for t in test['text'].values])
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

Preprocesando
Train: 40,800 | Val: 10,200 | Test: 8,500


## Entrega N3: Random Forest + BoW con RandomizedSearch (20 iter)

Las entregas anteriores usaron TF-IDF como representación del texto. En esta entrega
se reemplaza por **Bag of Words (BoW)**, que simplemente cuenta cuántas veces aparece
cada palabra en cada reseña, sin aplicar la penalización por frecuencia de documento
que hace TF-IDF.

La motivación es comparar si BoW aporta algo distinto a TF-IDF para Random Forest.
A diferencia de Naive Bayes (donde BoW mejoró el F1 de 0.6061 a 0.6533), en RF
el vectorizador es solo una etapa previa: el árbol puede ignorar features poco
informativas por sí solo, por lo que el impacto del cambio puede ser menor.

Se mantienen los mismos hiperparámetros explorados y la misma cantidad de iteraciones
(20) que en la N2 para que la comparación sea directa.

In [11]:
pipe = Pipeline([
    ("bow", make_bow()),
    ("rf",  RandomForestClassifier(class_weight="balanced", random_state=SEED, n_jobs=-1))])

param_rs_bow = {
    "bow__ngram_range":     [(1, 1), (1, 2)],
    "bow__min_df":          [3, 5, 10],
    "rf__n_estimators":     [100, 200, 300, 400],
    "rf__max_depth":        [None, 20, 50],
    "rf__min_samples_leaf": [1, 2, 4],
    "rf__max_features":     ["sqrt", "log2"],
}

rs_rf_bow = RandomizedSearchCV(
    pipe,
    param_rs_bow,
    n_iter=20,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    random_state=SEED,
    verbose=1
)

print("Buscando mejores hiperparámetros RF + BoW")
rs_rf_bow.fit(X_train, y_train)

print(f"\nMejores hiperparámetros: {rs_rf_bow.best_params_}")
print(f"Mejor F1-macro en CV: {rs_rf_bow.best_score_:.4f}")

y_pred = rs_rf_bow.best_estimator_.predict(X_val)
evaluate("rf_bow_randomsearch_v3", y_val, y_pred, hyperparams=rs_rf_bow.best_params_)

best_pipe = rs_rf_bow.best_estimator_
best_pipe.fit(np.concatenate([X_train, X_val]), np.concatenate([y_train, y_val]))
save_model(best_pipe, "rf_bow_randomsearch_v3")
make_submission(best_pipe, pd.DataFrame({"id": test["id"], "text": X_test}), "submission_rf_bow_randomsearch_v3.csv");

Buscando mejores hiperparámetros RF + BoW
Fitting 5 folds for each of 20 candidates, totalling 100 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Mejores hiperparámetros: {'rf__n_estimators': 300, 'rf__min_samples_leaf': 4, 'rf__max_features': 'log2', 'rf__max_depth': None, 'bow__ngram_range': (1, 2), 'bow__min_df': 10}
Mejor F1-macro en CV: 0.6402

=== rf_bow_randomsearch_v3 ===
Hiperparámetros: {'rf__n_estimators': 300, 'rf__min_samples_leaf': 4, 'rf__max_features': 'log2', 'rf__max_depth': None, 'bow__ngram_range': (1, 2), 'bow__min_df': 10}

F1-macro:  0.6402
Precision: 0.6401
Recall:    0.6404
Accuracy:  0.6853

              precision    recall  f1-score   support

    negativa     0.7461    0.7355    0.7408      4080
      neutra     0.4022    0.4162    0.4091      2040
    positiva     0.7721    0.7696    0.7708      4080

    accuracy                         0.6853     10200
   macro avg     0.6401    0.6404    0.6402     10200
weighted avg     0.6877    0.6853    0.6865     10200

Matriz de confusión (filas=real, cols=predicho):
          negativa  neutra  positiva
negativa      3001     704       375
neutra         6